# IOAI — 2026 Summer National Shredded Documents (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('파쇄 띠 데이터(data_documents.zip)는 노트북 첫 셀이 gdown 으로 자동 다운로드합니다. easyocr 도 자체 설치.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Shredded Documents — 모범답안 (Stage 1: reconstruction)

> **HAIO 2026 — Summer National Finals (CV) · Stage 1 = 30 pts**

Reassemble the shredded **citizen list** and read the records. Each 20px strip carries a 44px **11-bit barcode** (black=1, 4px/bit) giving its position; the body **background colour** identifies the document. Order strips by the barcode value, concatenate 62 per page, OCR, and read the 5 target citizens (7, 12, 21, 35, 42). **Submit** `submission.csv` (`id, first_name, last_name, date_of_birth`).

In [ ]:
import os, glob, subprocess, sys, zipfile, numpy as np
from PIL import Image
def _pip(p):
    subprocess.run([sys.executable,'-m','pip','install','-q',p],check=False)
try:
    import easyocr
except ImportError:
    _pip('easyocr'); import easyocr
# download + unzip the shred data (organisers' Google Drive, still live)
if not glob.glob('data/shreds/*.png'):
    os.makedirs('data', exist_ok=True)
    try:
        import gdown
    except ImportError:
        _pip('gdown'); import gdown
    gdown.download(id='1jG4yj_SM5Zw-697iRMHKWKkvmI10DrLq', output='data/docs.zip', quiet=True)
    zipfile.ZipFile('data/docs.zip').extractall('data')
print('shred strips:', len(glob.glob('data/shreds/*.png')))

In [ ]:
# --- reconstruct the citizen list: decode each strip's 11-bit barcode, order, concatenate ---
GREEN=(235,253,232)   # citizen-list background (vs suspect-list pink / briefing yellow)
strips={}
for p in sorted(glob.glob('data/shreds/*.png')):
    im=np.array(Image.open(p).convert('RGB'))
    if tuple(np.median(im[60:].reshape(-1,3),axis=0).astype(int))!=GREEN: continue
    bar=im[:44].mean(axis=(1,2))                       # 44px artifact = 11 bits x 4px (black=1)
    val=int(''.join('1' if bar[b*4:b*4+4].mean()<128 else '0' for b in range(11)),2)
    strips[val]=im[44:]                                # drop the barcode band
npages=len(strips)//62                                 # 62 columns (20px) per page
def page(pg): return np.concatenate([strips[v] for v in range(pg*62,(pg+1)*62)],axis=1)
print('citizen-list strips:', len(strips), '| pages:', npages)

In [ ]:
reader=easyocr.Reader(['en'])
cy=lambda b: float(np.array(b)[:,1].mean()); cx=lambda b: float(np.array(b)[:,0].mean())
norm=lambda t: t.strip().rstrip(':').upper()
records=[]
for pg in range(npages):
    items=reader.readtext(page(pg))
    def val_right(lb,maxdy=18):
        r=cy(lb); cs=sorted((cx(b),t) for b,t,c in items if abs(cy(b)-r)<maxdy and cx(b)>cx(lb)+5)
        return cs[0][1] if cs else ''
    fn=sorted((cy(b),b) for b,t,c in items if norm(t)=='FIRST NAME')
    for k,(top,_) in enumerate(fn):
        bot=fn[k+1][0] if k+1<len(fn) else 1e9
        blk=[(b,t,c) for b,t,c in items if top-5<=cy(b)<bot]
        def fld(name):
            L=[b for b,t,c in blk if norm(t)==name]; return val_right(L[0]) if L else ''
        records.append({'first':fld('FIRST NAME'),'last':fld('LAST NAME'),'dob':fld('DATE OF BIRTH')})
# records are globally in ID order -> record[id-1] is citizen `id`
print('parsed records:', len(records))

In [ ]:
import csv
targets=[7,12,21,35,42]
rows=[{'id':cid,'first_name':records[cid-1]['first'],'last_name':records[cid-1]['last'],
       'date_of_birth':records[cid-1]['dob']} for cid in targets]
with open('submission.csv','w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['id','first_name','last_name','date_of_birth']); w.writeheader(); w.writerows(rows)
print('wrote submission.csv'); [print(r) for r in rows]

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)